# 5 - TRAIN / VAL SPLIT, stratified by question type

The split becomes mandatory here. Every retrieval and reader measurement so far tunes
nothing, so reporting on all 16,229 questions was honest. Fine-tuning a reranker means
**selecting** - epochs, learning rate, negatives per question, which checkpoint - and
selecting on the same questions you report on inflates the number by an unknown amount.
Unlike the cleaning defects, that cannot be repaired afterwards.

Runs **locally, no GPU**: it needs `clean_corpus.pkl` (510 MB) and
`gold_chunk_rank.npy` (0.1 MB) from HF, and a minute of CPU.

## Two design choices, both deliberate

**1. Stratified by question type.** Question type correlates with answer type (dates,
people, places) and with difficulty, so a plain random split can leave a whole type thin in
val and make its error bar meaningless.

**2. Grouped by page, so no page appears in both splits.** A reranker is trained on
(query, chunk) pairs. If a page's chunks appear in training *and* validation, the model can
learn page-specific surface features and val will flatter it. Most pages carry one
question so this costs almost nothing, but it closes a real leak.

These two pull against each other - group assignment makes per-type quotas approximate -
so §4 reports the realised fraction per type rather than assuming it hit the target.

## Validation of the split itself

§5 checks balance on covariates that were **not** stratified on: answer position in the
page, page length, number of answer spans, and - most importantly - the **gold chunk's
retrieval rank**, which is a direct difficulty proxy. If val is accidentally easier or
harder than train on that axis, every later comparison is skewed.

In [ ]:
# §0 SETUP. NQ_RUN_ID is read at IMPORT time, so it is set before nqrun is imported.
import os, sys, json, pickle, re, collections
os.environ.setdefault('NQ_RUN_ID', 'split')
# Pushing is explicit and safe here: a new path, a few hundred KB, and pod runs need it.
# (An earlier blanket push overwrote a real artifact with a shard-0 one.)
os.environ['NQ_HF_PUSH'] = '1'
sys.path.insert(0, os.getcwd())
import numpy as np
import nqrun

In [ ]:
# §1 CONFIG
VAL_FRAC = 0.10          # ~1,623 of 16,229 questions
SEED     = 0
LEAD     = 400           # for the answer-in-lead balance report (the lead-only baseline's window)

N_PAGES_EXPECTED, N_QS_EXPECTED = 18997, 16229
OUT, HF_DIR = nqrun.art('step4-rerank'), 'step4-rerank'
print(f'val fraction {VAL_FRAC:.0%} | seed {SEED} | out {OUT}')
print(f'root {nqrun.ROOT} | run {nqrun.RUN_ID} | results -> {nqrun.RESULTS_DIR}')

## Definitions

Nothing below executes; running these cells only defines names.

In [ ]:
# DEFINITIONS
#
# NQ questions are lowercase and mostly unpunctuated, so the type is taken from
# the leading word(s). 'other' is a real category here: NQ contains declarative
# queries like 'the invention of linear perspective is generally attributed to'.
def qtype(q):
    """Question type from its leading word; anything else becomes 'other'."""
    w = WORD.findall(q.lower())
    if not w:
        return 'other'
    return FIRST.get(w[0], 'other')

def qsubtype(q):
    """Reported only, not stratified on: 'how many' behaves nothing like 'how long'."""
    w = WORD.findall(q.lower())
    if len(w) >= 2 and w[0] == 'how':
        return f'how {w[1]}'
    return qtype(q)

def page_type(h):
    """A page's type: the majority question type on it, first question breaking ties."""
    c = collections.Counter(types[i] for i in by_page[h])
    return max(c, key=lambda t: (c[t], t == types[by_page[h][0]]))

def compare(name, mask_fn):
    """Print one covariate as train share, val share and the gap."""
    a, b = mask_fn(train_idx), mask_fn(val_idx)
    print(f'  {name:<34} train {a:>6.1%} | val {b:>6.1%} | diff {(b - a) * 100:>+5.1f}')

## Orchestration

In [ ]:
# §2 LOAD + CLASSIFY QUESTION TYPE.
# NQ questions are lowercase and mostly unpunctuated. Type is taken from the leading
# word(s); anything that does not start with an interrogative lands in 'other', which is a
# real category here - NQ contains declarative queries like
# 'the invention of linear perspective is generally attributed to'.
C = pickle.load(open(nqrun.hf_pull('step1-clean/clean_corpus.pkl'), 'rb'))
docs, questions = C['documents'], C['questions']
assert len(docs) == N_PAGES_EXPECTED, f'{len(docs)} pages - WRONG ARTIFACT'
assert len(questions) == N_QS_EXPECTED, f'{len(questions)} questions - WRONG ARTIFACT'

WORD = re.compile(r"[a-z0-9']+")
FIRST = {'when': 'when', 'who': 'who', 'whose': 'who', 'whom': 'who',
         'what': 'what', 'whats': 'what', 'where': 'where', 'which': 'which',
         'how': 'how', 'why': 'why', 'name': 'name'}

types = np.array([qtype(q['question']) for q in questions])
TYPES = sorted(set(types.tolist()))
cnt = collections.Counter(types.tolist())
print(f"{'type':<10}{'n':>8}{'share':>9}")
for t in sorted(cnt, key=lambda x: -cnt[x]):
    print(f'{t:<10}{cnt[t]:>8,}{cnt[t] / len(types):>9.1%}')

sub = collections.Counter(qsubtype(q['question']) for q in questions)
print('\n"how" subtypes (reported, not stratified):')
for t, n in sub.most_common():
    if t.startswith('how '):
        print(f'  {t:<14}{n:>7,}')
print('\nexamples of "other":')
for q in [q['question'] for q in questions if qtype(q['question']) == 'other'][:5]:
    print(f'  {q[:72]}')

In [ ]:
# §3 SPLIT - stratified by type, grouped by page.
# Pages are the unit assigned, so a page never straddles the split. A page's type is its
# majority question type (ties broken by the first question), since a handful of pages
# carry questions of different types.
by_page = collections.defaultdict(list)
for i, q in enumerate(questions):
    by_page[q['doc_hash']].append(i)
pages = list(by_page)
print(f'{len(questions):,} questions over {len(pages):,} distinct gold pages '
      f'({len(questions) / len(pages):.2f} per page, '
      f'max {max(len(v) for v in by_page.values())})')

ptype = {h: page_type(h) for h in pages}
rng = np.random.default_rng(SEED)
val_pages = set()
for t in TYPES:
    group = [h for h in pages if ptype[h] == t]
    rng.shuffle(group)
    quota = VAL_FRAC * sum(len(by_page[h]) for h in group)   # quota in QUESTIONS
    taken = 0
    for h in group:
        if taken >= quota:
            break
        val_pages.add(h)
        taken += len(by_page[h])

val_idx = np.array(sorted(i for h in val_pages for i in by_page[h]))
train_idx = np.array(sorted(set(range(len(questions))) - set(val_idx.tolist())))
print(f'\ntrain {len(train_idx):,} | val {len(val_idx):,} '
      f'({len(val_idx) / len(questions):.2%} of questions)')

# ---- checks that can fail
tp = {questions[i]['doc_hash'] for i in train_idx}
vp = {questions[i]['doc_hash'] for i in val_idx}
print('\nCHECKS')
print(f'  question overlap           : {len(set(train_idx) & set(val_idx))}        <- must be 0')
print(f'  PAGE overlap               : {len(tp & vp)}        <- must be 0')
print(f'  train + val == all         : {len(train_idx) + len(val_idx) == len(questions)}')
assert not set(train_idx.tolist()) & set(val_idx.tolist())
assert not (tp & vp), 'a page appears in both splits - reranker leakage'
assert len(train_idx) + len(val_idx) == len(questions)

In [ ]:
# §4 PER-TYPE REALISATION. Group assignment makes the quota approximate, so the realised
# fraction is reported rather than assumed. The right-hand column is what matters for
# fine-tuning: enough TRAINING examples of every type.
print(f"{'type':<10}{'train':>9}{'val':>8}{'val %':>8}{'target':>8}   enough to train on?")
strata = {}
for t in sorted(TYPES, key=lambda x: -cnt[x]):
    ntr = int((types[train_idx] == t).sum())
    nva = int((types[val_idx] == t).sum())
    frac = nva / max(ntr + nva, 1)
    strata[t] = {'train': ntr, 'val': nva, 'val_frac': frac}
    flag = 'yes' if ntr >= 300 else ('thin' if ntr >= 100 else 'TOO FEW')
    print(f'{t:<10}{ntr:>9,}{nva:>8,}{frac:>8.1%}{VAL_FRAC:>8.0%}   {flag}')

worst = max(strata, key=lambda t: abs(strata[t]['val_frac'] - VAL_FRAC))
print(f"\nlargest deviation from the {VAL_FRAC:.0%} target: {worst} at "
      f"{strata[worst]['val_frac']:.1%}")
print('(deviation comes from assigning whole pages; a page carrying 3 questions can only\n'
      ' move all 3 at once)')

In [ ]:
# §5 BALANCE ON COVARIATES WE DID *NOT* STRATIFY ON.
# If val is accidentally easier or harder than train, every later comparison is skewed.
# The last block is the one that matters: the gold chunk's retrieval rank is a direct
# difficulty proxy, straight from 2c.
pos = np.array([q['gold'][0][1] for q in questions])
plen = np.array([len(docs[q['doc_hash']]['words']) for q in questions])
nspan = np.array([len(q['gold']) for q in questions])

print('BALANCE')
compare(f'answer inside first {LEAD} words', lambda ix: float((pos[ix] < LEAD).mean()))
compare('page under 1,000 words', lambda ix: float((plen[ix] < 1000).mean()))
compare('page over 4,000 words', lambda ix: float((plen[ix] >= 4000).mean()))
compare('more than one answer span', lambda ix: float((nspan[ix] > 1).mean()))
print(f"  {'median page length':<34} train {int(np.median(plen[train_idx])):>6,} | "
      f'val {int(np.median(plen[val_idx])):>6,}')

# difficulty proxy from notebook 4 - skipped gracefully if the artifact is unavailable
try:
    crank = np.load(nqrun.hf_pull('step2-retrieval/gold_chunk_rank.npy'))
    assert len(crank) == len(questions)
    print('\nDIFFICULTY (gold chunk retrieval rank, from notebook 4)')
    for lbl, lo, hi in [('rank 0 (already top-1)', 0, 1), ('rank 1-4', 1, 5),
                        ('rank 5-19', 5, 20), ('rank 20-99', 20, 100),
                        ('outside top-100', 100, 10 ** 9)]:
        a = float(((crank[train_idx] >= lo) & (crank[train_idx] < hi)).mean())
        b = float(((crank[val_idx] >= lo) & (crank[val_idx] < hi)).mean())
        print(f'  {lbl:<34} train {a:>6.1%} | val {b:>6.1%} | diff {(b - a) * 100:>+5.1f}')
    print(f"  {'median gold chunk rank':<34} "
          f'train {int(np.median(crank[train_idx])):>6,} | '
          f'val {int(np.median(crank[val_idx])):>6,}')
    have_rank = True
except Exception as e:
    print(f'\nDIFFICULTY: skipped ({type(e).__name__}: {str(e)[:60]})')
    have_rank = False
print('\nAny |diff| above ~2 points is worth a second look - re-run with another SEED and\n'
      'compare, since nothing above was stratified on.')

In [ ]:
# §6 SAVE. Question INDICES into clean_corpus.pkl's `questions` list, plus doc_hash so the
# split can be re-derived if that list ever changes order.
payload = {
    'created_utc': __import__('datetime').datetime.now(
        __import__('datetime').timezone.utc).isoformat(timespec='seconds'),
    'source': 'step1-clean/clean_corpus.pkl',
    'n_questions': len(questions), 'n_pages': len(docs),
    'val_frac_target': VAL_FRAC, 'seed': SEED,
    'grouped_by': 'doc_hash', 'stratified_by': 'question_type',
    'types': TYPES, 'strata': strata,
    'train_idx': train_idx.tolist(), 'val_idx': val_idx.tolist(),
    'train_doc_hashes': sorted({questions[i]['doc_hash'] for i in train_idx}),
    'val_doc_hashes': sorted({questions[i]['doc_hash'] for i in val_idx}),
    'question_type': types.tolist(),
}
json.dump(payload, open(OUT / 'split.json', 'w'))
print(f"-> {OUT / 'split.json'} ({os.path.getsize(OUT / 'split.json') / 1e6:.2f} MB)")

nqrun.hf_push(str(OUT / 'split.json'), f'{HF_DIR}/split.json',
              f'train/val split: {len(train_idx)}/{len(val_idx)}, '
              f'stratified by question type, grouped by page, seed {SEED}')
nqrun.log_metrics('5_split', n_questions=len(questions), n_train=len(train_idx),
                  n_val=len(val_idx), val_frac=len(val_idx) / len(questions),
                  seed=SEED, n_types=len(TYPES),
                  grouped_by='doc_hash', stratified_by='question_type',
                  **{f'train_{t}': strata[t]['train'] for t in TYPES},
                  **{f'val_{t}': strata[t]['val'] for t in TYPES})